In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "Prior authorization is required for MRI procedures."

tokens = encoding.encode(text)

print("Original text:", text)
print("Word count:", len(text.split()))
print("Token count:", len(tokens))
print("Token IDs:", tokens)

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
model = os.getenv("AZURE_OPENAI_MODEL")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

print("Azure model client ready")

In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} -> {repr(token_text)}")

In [ ]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    token_ids = encoding.encode(text)

    print("\nTEXT:", text)
    print("Words :", len(text.split()))
    print("Tokens:", len(token_ids))

In [ ]:
samples = [
    "Prior authorization is required for MRI procedures.",
    "PA is required for MRI.",
    "The member's deductible is $1,500.",
    "AuthorizationRequired=True",
    "पूर्व प्राधिकरण आवश्यक है"
]

for text in samples:
    words = len(text.split())
    tokens = len(encoding.encode(text))
    ratio = tokens / words if words > 0 else 0

    print(f"\n{text}")
    print(f"Words            : {words}")
    print(f"Tokens           : {tokens}")
    print(f"Tokens per word  : {ratio:.2f}")

In [ ]:
prompt_text = "Explain prior authorization in healthcare in two sentences."

local_token_count = len(encoding.encode(prompt_text))

api_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": prompt_text
        }
    ]
)

print("Local text token estimate :", local_token_count)
print("Azure prompt tokens       :", api_response.usage.prompt_tokens)

In [ ]:
short_context = """
Policy: MRI procedures require prior authorization.
"""

long_context = """
Policy: MRI procedures require prior authorization.
CT scans require prior authorization only for outpatient procedures.
Emergency room imaging does not require prior authorization.
Physical therapy requires authorization after 10 visits.
Specialist consultations do not require prior authorization.
"""

question = "Does an MRI require prior authorization?"

In [ ]:
short_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{short_context}

Question:
{question}
"""
        }
    ]
)

print(short_response.choices[0].message.content)
print("\nPrompt tokens:", short_response.usage.prompt_tokens)

In [ ]:
long_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{long_context}

Question:
{question}
"""
        }
    ]
)

print(long_response.choices[0].message.content)
print("\nPrompt tokens:", long_response.usage.prompt_tokens)

In [ ]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

In [ ]:
short_tokens = short_response.usage.prompt_tokens
long_tokens = long_response.usage.prompt_tokens

increase = long_tokens - short_tokens
increase_pct = (increase / short_tokens) * 100

print("Short-context tokens :", short_tokens)
print("Long-context tokens  :", long_tokens)
print("Additional tokens    :", increase)
print(f"Increase             : {increase_pct:.1f}%")

In [ ]:
noisy_context = """
Member ID: M102938
Plan Type: PPO
Primary Care Copay: $25
Specialist Copay: $50
Emergency Room Copay: $250
Dental coverage is not included.
Vision coverage is included once every 24 months.
Physical therapy requires authorization after 10 visits.
MRI procedures require prior authorization.
Member mailing address was updated last month.
Claims are processed within standard turnaround time.
"""

question = "Does an MRI require prior authorization?"

In [ ]:
noisy_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "Answer only from the provided policy context."
        },
        {
            "role": "user",
            "content": f"""
Context:
{noisy_context}

Question:
{question}
"""
        }
    ]
)

print(noisy_response.choices[0].message.content)
print("\nPrompt tokens:", noisy_response.usage.prompt_tokens)

In [ ]:
print("Short context tokens :", short_response.usage.prompt_tokens)
print("Long context tokens  :", long_response.usage.prompt_tokens)
print("Noisy context tokens :", noisy_response.usage.prompt_tokens)

In [ ]:
import pandas as pd

context_comparison = pd.DataFrame([
    {
        "Scenario": "Short context",
        "Prompt Tokens": short_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "High"
    },
    {
        "Scenario": "Long context",
        "Prompt Tokens": long_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Medium"
    },
    {
        "Scenario": "Noisy context",
        "Prompt Tokens": noisy_response.usage.prompt_tokens,
        "Answer Quality": "Correct",
        "Context Relevance": "Low"
    }
])

context_comparison

In [ ]:
embedding_samples = [
    "Prior authorization is required for MRI procedures.",
    "MRI scans need approval from the health insurance payer.",
    "The member updated their mailing address.",
    "The deductible must be paid before the health plan starts sharing costs."
]

for i, text in enumerate(embedding_samples, start=1):
    print(f"{i}. {text}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

In [ ]:
embedding_model = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

print("Embedding model configured:", bool(embedding_model))
print("Embedding deployment       :", embedding_model)

In [ ]:
embedding_response = client.embeddings.create(
    model=embedding_model,
    input=embedding_samples
)

embeddings = [item.embedding for item in embedding_response.data]

print("Number of embeddings:", len(embeddings))
print("Embedding dimension :", len(embeddings[0]))

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )

for i in range(len(embedding_samples)):
    for j in range(i + 1, len(embedding_samples)):
        score = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

        print(f"{i+1} vs {j+1}: {score:.4f}")

In [ ]:
import pandas as pd
import numpy as np

similarity_matrix = np.zeros((len(embedding_samples), len(embedding_samples)))

for i in range(len(embedding_samples)):
    for j in range(len(embedding_samples)):
        similarity_matrix[i][j] = cosine_similarity(
            embeddings[i],
            embeddings[j]
        )

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=[f"Text {i+1}" for i in range(len(embedding_samples))],
    columns=[f"Text {i+1}" for i in range(len(embedding_samples))]
)

similarity_df.round(3)

In [ ]:
reasoning_case = """
A member has already completed 8 physical therapy visits.
The health plan policy allows 10 visits without prior authorization.
Prior authorization is required starting from the 11th visit.

The provider is requesting the member's 9th physical therapy visit.
"""

reasoning_question = """
Does this visit require prior authorization?
Give only the final decision and one-line justification.
"""

print(reasoning_case)
print(reasoning_question)

In [ ]:
direct_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{reasoning_case}

{reasoning_question}
"""
        }
    ]
)

print(direct_reasoning_response.choices[0].message.content)

In [ ]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

In [ ]:
structured_reasoning_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy reasoning assistant.

Evaluate the case using this structure:
1. Policy rule
2. Case fact
3. Decision

Keep the answer concise.
"""
        },
        {
            "role": "user",
            "content": f"""
{reasoning_case}

Question:
Does this visit require prior authorization?
"""
        }
    ]
)

print(structured_reasoning_response.choices[0].message.content)

In [ ]:
ambiguous_case = """
A member has completed 10 physical therapy visits.

The policy states:
"Prior authorization may be required after the initial covered visits,
depending on the member's plan and clinical review requirements."

The provider is requesting the 11th visit.
"""

ambiguous_question = """
Does the 11th visit require prior authorization?
"""

print(ambiguous_case)
print(ambiguous_question)

In [ ]:
ambiguous_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

{ambiguous_question}
Give a clear decision and a short justification.
"""
        }
    ]
)

print(ambiguous_response.choices[0].message.content)

In [ ]:
guardrailed_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only from the provided policy.
2. Do not infer missing policy conditions.
3. If the available information is insufficient for a definitive decision,
   clearly state "Insufficient information".
4. Explain what additional information is required.
"""
        },
        {
            "role": "user",
            "content": f"""
{ambiguous_case}

Question:
Does the 11th visit require prior authorization?
"""
        }
    ]
)

print(guardrailed_response.choices[0].message.content)

In [ ]:
print("WITHOUT GUARDRAIL")
print("-" * 50)
print(ambiguous_response.choices[0].message.content)

print("\nWITH GUARDRAIL")
print("-" * 50)
print(guardrailed_response.choices[0].message.content)

In [ ]:
hallucination_question = """
According to the ZS Platinum Plus Health Plan 2026,
what is the maximum number of chiropractic visits allowed per year?
"""

print(hallucination_question)

In [ ]:
hallucination_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_response.choices[0].message.content)

In [ ]:
hallucination_guardrail_response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": """
You are a healthcare payer policy assistant.

Rules:
1. Answer only when the required information is present in the provided context.
2. Never invent plan benefits, limits, policies, or coverage rules.
3. If no supporting policy context is provided, respond:
   "I cannot determine this from the available information."
"""
        },
        {
            "role": "user",
            "content": hallucination_question
        }
    ]
)

print(hallucination_guardrail_response.choices[0].message.content)

In [ ]:
print("WITHOUT EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_response.choices[0].message.content)

print("\nWITH EVIDENCE GUARDRAIL")
print("-" * 50)
print(hallucination_guardrail_response.choices[0].message.content)

In [ ]:
load_dotenv(".env", override=True)

model_primary = os.getenv("AZURE_OPENAI_MODEL")
model_secondary = os.getenv("AZURE_OPENAI_MODEL_SECONDARY")

print("Primary model   :", model_primary)
print("Secondary model :", model_secondary)

In [ ]:
comparison_prompt = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Answer using exactly this format:

Decision:
Reason:
"""

In [ ]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

In [ ]:
comparison_prompt_guarded = """
A member has completed 10 physical therapy visits.

Policy:
Prior authorization is required starting from the 11th visit.

Current request:
The provider is requesting the member's 11th physical therapy visit.

Rules:
1. Use only the information provided.
2. Do not assume whether authorization has already been requested, approved, or denied.
3. If the information is insufficient for an approval/denial decision, state that clearly.

Answer using exactly this format:

Prior Authorization Required:
Approval Decision:
Reason:
"""

In [ ]:
primary_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

secondary_response = client.chat.completions.create(
    model=model_secondary,
    messages=[
        {
            "role": "user",
            "content": comparison_prompt
        }
    ]
)

print("PRIMARY MODEL:", model_primary)
print("-" * 60)
print(primary_response.choices[0].message.content)

print("\nSECONDARY MODEL:", model_secondary)
print("-" * 60)
print(secondary_response.choices[0].message.content)

In [ ]:
def show_metrics(name, response):
    print(name)
    print("-" * 50)

    print("Actual model       :", response.model)
    print("Prompt tokens      :", response.usage.prompt_tokens)
    print("Completion tokens  :", response.usage.completion_tokens)
    print("Total tokens       :", response.usage.total_tokens)

    latency = getattr(response.usage, "latency_checkpoint", None)

    if latency:
        print("Total latency (ms) :", latency.get("total_duration_ms"))
        print("First token (ms)   :", latency.get("user_visible_ttft_ms"))

    print()


show_metrics("GPT-4.1-mini", primary_response)
show_metrics("GPT-5-mini", secondary_response)

In [ ]:
print("GPT-4.1-mini token details")
print(primary_response.usage.completion_tokens_details)

print("\nGPT-5-mini token details")
print(secondary_response.usage.completion_tokens_details)

In [ ]:
model_comparison = pd.DataFrame([
    {
        "Model": "GPT-4.1-mini",
        "Prompt Tokens": primary_response.usage.prompt_tokens,
        "Completion Tokens": primary_response.usage.completion_tokens,
        "Reasoning Tokens": primary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": primary_response.usage.total_tokens,
        "Latency (ms)": primary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Fast, but made unsupported approval inference"
    },
    {
        "Model": "GPT-5-mini",
        "Prompt Tokens": secondary_response.usage.prompt_tokens,
        "Completion Tokens": secondary_response.usage.completion_tokens,
        "Reasoning Tokens": secondary_response.usage.completion_tokens_details.reasoning_tokens,
        "Total Tokens": secondary_response.usage.total_tokens,
        "Latency (ms)": secondary_response.usage.latency_checkpoint["total_duration_ms"],
        "Observed Behavior": "Better evidence discipline, but higher reasoning cost"
    }
])

model_comparison

In [ ]:
weak_prompt = """
Explain prior authorization.
"""

weak_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": weak_prompt
        }
    ]
)

print(weak_prompt_response.choices[0].message.content)
print("\nTotal tokens:", weak_prompt_response.usage.total_tokens)

In [ ]:
strong_prompt = """
You are a healthcare payer domain assistant.

Explain prior authorization to a healthcare technology professional.

Requirements:
1. Use payer terminology.
2. Explain the purpose and workflow.
3. Keep the answer to exactly 3 bullet points.
4. Maximum 80 words.
5. Do not add information beyond the requested scope.
"""

strong_prompt_response = client.chat.completions.create(
    model=model_primary,
    messages=[
        {
            "role": "user",
            "content": strong_prompt
        }
    ]
)

print(strong_prompt_response.choices[0].message.content)
print("\nTotal tokens:", strong_prompt_response.usage.total_tokens)

In [ ]:
print("WEAK PROMPT")
print("Prompt tokens     :", weak_prompt_response.usage.prompt_tokens)
print("Completion tokens :", weak_prompt_response.usage.completion_tokens)
print("Total tokens      :", weak_prompt_response.usage.total_tokens)

print("\nSTRONG PROMPT")
print("Prompt tokens     :", strong_prompt_response.usage.prompt_tokens)
print("Completion tokens :", strong_prompt_response.usage.completion_tokens)
print("Total tokens      :", strong_prompt_response.usage.total_tokens)

In [ ]:
prompt_comparison = pd.DataFrame([
    {
        "Prompt Type": "Weak",
        "Prompt Tokens": weak_prompt_response.usage.prompt_tokens,
        "Completion Tokens": weak_prompt_response.usage.completion_tokens,
        "Total Tokens": weak_prompt_response.usage.total_tokens,
        "Control Level": "Low",
        "Output Structure": "Uncontrolled"
    },
    {
        "Prompt Type": "Strong",
        "Prompt Tokens": strong_prompt_response.usage.prompt_tokens,
        "Completion Tokens": strong_prompt_response.usage.completion_tokens,
        "Total Tokens": strong_prompt_response.usage.total_tokens,
        "Control Level": "High",
        "Output Structure": "Controlled"
    }
])

prompt_comparison
